## Compaction in action

In [1]:
from pyspark.sql import SparkSession
import os
import json
from pprint import pprint
warehouse_path = r"C:\iceberg-warehouse"

print(os.listdir(warehouse_path))

['db', 'demo']


In [2]:
spark = SparkSession.builder \
    .appName("IcebergLocal") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    ) \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "file:///C:/iceberg-warehouse"
    ) \
    .getOrCreate()

In [15]:
spark.sql("""

CREATE TABLE local.demo.comp (
    id BIGINT,
    load_id STRING,
    value STRING
)
USING iceberg
PARTITIONED BY (load_id)
""")

DataFrame[]

In [16]:


spark.sql("""
insert into local.demo.comp values
(1,"1","A"),
(2,"1","A")
""")

DataFrame[]

In [17]:
spark.sql("""
select * from local.demo.comp.snapshots

""").collect()

[Row(committed_at=datetime.datetime(2026, 6, 6, 21, 53, 21, 124000), snapshot_id=3636983554533033969, parent_id=None, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/comp/metadata/snap-3636983554533033969-1-2efa0acd-c974-4cd3-a5ee-36a2415f23a1.avro', summary={'spark.app.id': 'local-1780762556332', 'changed-partition-count': '1', 'added-data-files': '1', 'total-equality-deletes': '0', 'added-records': '2', 'total-position-deletes': '0', 'added-files-size': '937', 'total-delete-files': '0', 'total-files-size': '937', 'total-records': '2', 'total-data-files': '1'})]

In [18]:
spark.sql("""

INSERT INTO local.demo.comp VALUES
(3,'2','C'),
(4,'2','D')
""")

DataFrame[]

In [19]:
spark.sql("""

INSERT INTO local.demo.comp VALUES
(5,'3','E'),
(6,'3','F')

""")

DataFrame[]

In [21]:
spark.sql("""
SELECT *
FROM local.demo.comp.files
""").collect()

[Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/comp/data/load_id=3/00000-17-052c7b0f-5b03-4bb9-ac33-062a2a5f142e-0-00001.parquet', file_format='PARQUET', spec_id=0, partition=Row(load_id='3'), record_count=2, file_size_in_bytes=896, column_sizes={1: 48, 2: 67, 3: 42}, value_counts={1: 2, 2: 2, 3: 2}, null_value_counts={1: 0, 2: 0, 3: 0}, nan_value_counts={}, lower_bounds={1: bytearray(b'\x05\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'3'), 3: bytearray(b'E')}, upper_bounds={1: bytearray(b'\x06\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'3'), 3: bytearray(b'F')}, key_metadata=None, split_offsets=[4], equality_ids=None, sort_order_id=0, readable_metrics=Row(id=Row(column_size=48, value_count=2, null_value_count=0, nan_value_count=None, lower_bound=5, upper_bound=6), load_id=Row(column_size=67, value_count=2, null_value_count=0, nan_value_count=None, lower_bound='3', upper_bound='3'), value=Row(column_size=42, value_count=2, null_value_count=0, nan_value_count=None, lower_

In [23]:
spark.sql("""
SELECT *
FROM local.demo.members.history
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-06-06 21:30:25.007|1182519879785584587|NULL               |true               |
|2026-06-06 21:33:37.652|4426681822208192271|1182519879785584587|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [24]:
spark.sql("""
CALL local.system.rewrite_data_files(
    table => 'demo.comp'
)
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|0                         |0                     |0                    |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



In [25]:
spark.sql("""
SELECT
    file_path,
    record_count,
    file_size_in_bytes
FROM local.demo.comp.files
""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                        |record_count|file_size_in_bytes|
+-----------------------------------------------------------------------------------------------------------------+------------+------------------+
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=3/00000-17-052c7b0f-5b03-4bb9-ac33-062a2a5f142e-0-00001.parquet|2           |896               |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=2/00000-14-0647e879-8bb3-4e56-b9c1-73cc7d8c9e70-0-00001.parquet|2           |896               |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=1/00000-10-281edb6b-f5af-45ec-9f0f-042d2b3b8be5-0-00001.parquet|2           |937               |
+---------------------------------------------------------------------------------------------------------------

In [26]:
"""
| partition | files |
| --------- | ----- |
| load_id=1 | 1     |
| load_id=2 | 1     |
| load_id=3 | 1     |


Iceberg compaction works within partitions.
Iceberg compaction works within partitions.

There is nothing to merge because:
partition load_id=1 has only 1 file
partition load_id=2 has only 1 file
partition load_id=3 has only 1 file
Compaction says:

I need multiple files in the same partition to combine.
"""

'\n| partition | files |\n| --------- | ----- |\n| load_id=1 | 1     |\n| load_id=2 | 1     |\n| load_id=3 | 1     |\n\n\nIceberg compaction works within partitions.\nIceberg compaction works within partitions.\n\nThere is nothing to merge because:\npartition load_id=1 has only 1 file\npartition load_id=2 has only 1 file\npartition load_id=3 has only 1 file\nCompaction says:\n\nI need multiple files in the same partition to combine.\n'

In [27]:
spark.sql("""
INSERT INTO local.demo.comp VALUES
(7,'1','G'),
(8,'1','H')
""")

DataFrame[]

In [28]:
spark.sql("""
INSERT INTO local.demo.comp VALUES
(9,'1','I'),
(10,'1','J')
""")

DataFrame[]

In [29]:
spark.sql("""
SELECT
    file_path,
    partition,
    record_count
FROM local.demo.comp.files
""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------+---------+------------+
|file_path                                                                                                        |partition|record_count|
+-----------------------------------------------------------------------------------------------------------------+---------+------------+
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=1/00000-26-8adcb90c-b380-47d0-b87f-68ad26bdecc0-0-00001.parquet|{1}      |2           |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=1/00000-23-8d1fe0c4-539f-4aac-bfe2-28c11887dad6-0-00001.parquet|{1}      |2           |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=3/00000-17-052c7b0f-5b03-4bb9-ac33-062a2a5f142e-0-00001.parquet|{3}      |2           |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=2/00000-14-0647e879-8bb3-4e56-b9c1-73cc7d8c9e70-0-00001.parquet|{2}      |2           |
|file:/C:/iceberg-warehouse

In [30]:
spark.sql("""
CALL local.system.rewrite_data_files(
  table => 'demo.comp',
  options => map(
    'min-input-files','2'
  )
)
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|3                         |1                     |2726                 |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



In [31]:
spark.sql("""
SELECT
    file_path,
    partition,
    record_count
FROM local.demo.comp.files
""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------+---------+------------+
|file_path                                                                                                        |partition|record_count|
+-----------------------------------------------------------------------------------------------------------------+---------+------------+
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=1/00000-29-c579cd1e-05a9-4439-a300-14f71c573602-0-00001.parquet|{1}      |6           |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=3/00000-17-052c7b0f-5b03-4bb9-ac33-062a2a5f142e-0-00001.parquet|{3}      |2           |
|file:/C:/iceberg-warehouse/demo/comp/data/load_id=2/00000-14-0647e879-8bb3-4e56-b9c1-73cc7d8c9e70-0-00001.parquet|{2}      |2           |
+-----------------------------------------------------------------------------------------------------------------+---------+------------+



In [33]:
spark.sql("""

select * from local.demo.comp.history

""").collect()

[Row(made_current_at=datetime.datetime(2026, 6, 6, 21, 53, 21, 124000), snapshot_id=3636983554533033969, parent_id=None, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 6, 6, 21, 53, 37, 224000), snapshot_id=703308302427061885, parent_id=3636983554533033969, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 6, 6, 21, 53, 57, 886000), snapshot_id=6694373711540834442, parent_id=703308302427061885, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 6, 6, 22, 1, 30, 211000), snapshot_id=1440324677463908206, parent_id=6694373711540834442, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 6, 6, 22, 1, 34, 312000), snapshot_id=4156204392769642720, parent_id=1440324677463908206, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 6, 6, 22, 2, 0, 842000), snapshot_id=7290499458090154921, parent_id=4156204392769642720, is_current_ancestor=True)]

In [34]:
spark.sql("""

select * from local.demo.comp.snapshots

""").collect()

[Row(committed_at=datetime.datetime(2026, 6, 6, 21, 53, 21, 124000), snapshot_id=3636983554533033969, parent_id=None, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/comp/metadata/snap-3636983554533033969-1-2efa0acd-c974-4cd3-a5ee-36a2415f23a1.avro', summary={'spark.app.id': 'local-1780762556332', 'changed-partition-count': '1', 'added-data-files': '1', 'total-equality-deletes': '0', 'added-records': '2', 'total-position-deletes': '0', 'added-files-size': '937', 'total-delete-files': '0', 'total-files-size': '937', 'total-records': '2', 'total-data-files': '1'}),
 Row(committed_at=datetime.datetime(2026, 6, 6, 21, 53, 37, 224000), snapshot_id=703308302427061885, parent_id=3636983554533033969, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/comp/metadata/snap-703308302427061885-1-e9b8d4fd-797d-4baf-8b53-f8ca12d35d59.avro', summary={'spark.app.id': 'local-1780762556332', 'changed-partition-count': '1', 'added-data-files': '1', 'total-equality-deletes

 Row(committed_at=datetime.datetime(2026, 6, 6, 22, 2, 0, 842000), snapshot_id=7290499458090154921, parent_id=4156204392769642720, operation='replace', manifest_list='file:/C:/iceberg-warehouse/demo/comp/metadata/snap-7290499458090154921-1-ca79172a-0d5f-451f-a6bf-daa0e2fa9b36.avro', summary={'added-data-files': '1', 'total-equality-deletes': '0', 'added-records': '6', 'deleted-data-files': '3', 'deleted-records': '6', 'total-records': '10', 'removed-files-size': '2726', 'changed-partition-count': '1', 'total-position-deletes': '0', 'added-files-size': '947', 'total-delete-files': '0', 'total-files-size': '2739', 'total-data-files': '3'})]

 so here the snapshot of the compaction performed with the operation replcae 


In [40]:
spark.sql("""
SELECT *
FROM local.demo.comp
WHERE load_id = '1' 
""").collect()

[Row(id=1, load_id='1', value='A'),
 Row(id=2, load_id='1', value='A'),
 Row(id=7, load_id='1', value='G'),
 Row(id=8, load_id='1', value='H'),
 Row(id=9, load_id='1', value='I'),
 Row(id=10, load_id='1', value='J')]

when we did compaction here we really did not free up the old file i.e delete them they are stll present in the fs, that snapshot is not deleted
yet, this allows for time travel

the storeage can be freed up only by expiration of snapshots and when garbage collection happens then 
there is no reference to the old snaps they gets deleted freeing up space 

In [41]:
spark.sql("""
SELECT
    made_current_at,
    snapshot_id,
    is_current_ancestor
FROM local.demo.comp.history
ORDER BY made_current_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |is_current_ancestor|
+-----------------------+-------------------+-------------------+
|2026-06-06 21:53:21.124|3636983554533033969|true               |
|2026-06-06 21:53:37.224|703308302427061885 |true               |
|2026-06-06 21:53:57.886|6694373711540834442|true               |
|2026-06-06 22:01:30.211|1440324677463908206|true               |
|2026-06-06 22:01:34.312|4156204392769642720|true               |
|2026-06-06 22:02:00.842|7290499458090154921|true               |
+-----------------------+-------------------+-------------------+



In [42]:
spark.sql("""
SELECT
    snapshot_id,
    operation,
    committed_at
FROM local.demo.comp.snapshots
ORDER BY committed_at
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|3636983554533033969|append   |2026-06-06 21:53:21.124|
|703308302427061885 |append   |2026-06-06 21:53:37.224|
|6694373711540834442|append   |2026-06-06 21:53:57.886|
|1440324677463908206|append   |2026-06-06 22:01:30.211|
|4156204392769642720|append   |2026-06-06 22:01:34.312|
|7290499458090154921|replace  |2026-06-06 22:02:00.842|
+-------------------+---------+-----------------------+



In [45]:
spark.sql("""
CALL local.system.expire_snapshots(
    table => 'demo.comp',
    retain_last => 1
)

""").collect()

[Row(deleted_data_files_count=0, deleted_position_delete_files_count=0, deleted_equality_delete_files_count=0, deleted_manifest_files_count=0, deleted_manifest_lists_count=0, deleted_statistics_files_count=0)]

nothing was expired due to retention p

In [46]:
spark.sql("""
SHOW TBLPROPERTIES local.demo.comp
""").show(truncate=False)

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |7290499458090154921|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|write.parquet.compression-codec|zstd               |
+-------------------------------+-------------------+



In [47]:
spark.sql("""
CALL local.system.expire_snapshots(
    table => 'demo.comp',
    older_than => TIMESTAMP '2100-01-01 00:00:00',
    retain_last => 1
)
""").show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|3                       |0                                  |0                                  |3                           |5                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [48]:
spark.sql("""
SELECT *
FROM local.demo.comp.history
ORDER BY made_current_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-06-06 22:02:00.842|7290499458090154921|4156204392769642720|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [49]:
spark.sql("""

SELECT *
FROM local.demo.comp
VERSION AS OF 4156204392769642720
""")

IllegalArgumentException: Cannot find snapshot with ID 4156204392769642720

In [50]:
spark.sql("""
SELECT *
FROM local.demo.comp
WHERE load_id = '1' 
""").collect()

[Row(id=1, load_id='1', value='A'),
 Row(id=2, load_id='1', value='A'),
 Row(id=7, load_id='1', value='G'),
 Row(id=8, load_id='1', value='H'),
 Row(id=9, load_id='1', value='I'),
 Row(id=10, load_id='1', value='J')]

In [51]:
spark.sql("""
SELECT *
FROM local.demo.comp
WHERE load_id = '2' 
""").collect()

[Row(id=3, load_id='2', value='C'), Row(id=4, load_id='2', value='D')]

In [52]:
spark.sql("""
SELECT *
FROM local.demo.comp
WHERE load_id = '3' 
""").collect()

[Row(id=5, load_id='3', value='E'), Row(id=6, load_id='3', value='F')]